# Overnight queue: context turns (E2) and the premise control

The baseline LSTM with 0, 1, 3 and 10 previous turns (`lstm-k5` is the 5). Ten turns need a longer character budget, and batches of 64 to fit. Last comes plain 768 with the premise on the same schedule as `lstm-attention-768-premise`, so the two differ only in attention. A failed run is reported and the queue moves on; the summary prints at the end.

Attach to **Serverless GPU, A10**. Runs alongside `lstm-queue-architectures` on a second GPU.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT))

from drzavni_posao.generation.lstm.tracking import run_queue
from drzavni_posao.generation.lstm.train import TrainConfig

In [ ]:
outcomes = run_queue([
    ("lstm-k0", TrainConfig(context_turns=0)),
    ("lstm-k1", TrainConfig(context_turns=1)),
    ("lstm-k3", TrainConfig(context_turns=3)),
    (
        "lstm-k10",
        TrainConfig(
            context_turns=10, max_context_chars=2500, batch_size=64, learning_rate=2e-3
        ),
    ),
    (
        "lstm-plain-768-premise-sched",
        TrainConfig(
            hidden_size=768,
            dropout=0.25,
            premise_mode="prefix",
            batch_size=64,
            learning_rate=1.5e-3,
            epochs=40,
            scheduler_patience=1,
            patience=5,
        ),
    ),
])